## Imports

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
import joblib
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv("cleared_dataset.csv")

target = "Retard moyen des trains en retard au départ"

df[target] = (
    df[target]
    .astype(str)
    .str.replace(",", ".", regex=False)
)

df[target] = pd.to_numeric(df[target], errors="coerce")

df = df.dropna(subset=[target])

features = [
    "Gare de départ",
    "Gare d'arrivée",
    "Service",
    "Nombre de circulations prévues",
    "Nombre de trains annulés"
]

X = df[features]
y = df[target]

X = pd.get_dummies(X, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

baseline_value = y_train.mean()
baseline_pred = np.full(len(y_test), baseline_value)

print("Baseline MAE:", mean_absolute_error(y_test, baseline_pred))
print("Baseline RMSE:", np.sqrt(mean_squared_error(y_test, baseline_pred)))
print("Baseline R2:", r2_score(y_test, baseline_pred))

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("\nLinear Regression Results")
print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2:", r2_score(y_test, y_pred))

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("\nLinear Regression Results")
print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2:", r2_score(y_test, y_pred))

rf = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

print("\nRandom Forest Results")
print("MAE:", mean_absolute_error(y_test, y_pred_rf))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_rf)))
print("R2:", r2_score(y_test, y_pred_rf))

joblib.dump(rf, "model.pkl")



Baseline MAE: 6.719094480529909
Baseline RMSE: 13.19284124111289
Baseline R2: -2.6889171333976947e-08

Linear Regression Results
MAE: 6.622251554697923
RMSE: 13.206678188761392
R2: -0.0020987718097216224

Linear Regression Results
MAE: 6.622251554697923
RMSE: 13.206678188761392
R2: -0.0020987718097216224

Random Forest Results
MAE: 6.17589201878705
RMSE: 13.52850798415494
R2: -0.051533574025487416


['model.pkl']

## Model Interpretation

The baseline model predicts the mean delay for all observations.
It serves as a reference to evaluate whether our models actually learn meaningful patterns.

The Linear Regression model slightly improves performance over the baseline, suggesting that some linear relationships exist between journey characteristics and delays.

The Random Forest model performs better than Linear Regression, with lower MAE and RMSE and a higher R² score. This indicates that train delays are influenced by non-linear interactions between features such as departure station, service type, and number of cancellations.

Therefore, Random Forest is selected as the final model for deployment in the dashboard.